# I22 batch processing with the MoDaCor runtime

This notebook is the ordinary, non-chunked I22 workflow. It preprocesses only incompatible frame metadata, creates one reusable runtime session per detector, and writes one result per measurement. Geometry and physical corrections remain in the tracked MoDaCor pipelines.

The three chunk-transport examples live in separate notebooks beside this one. Run from top to bottom and edit only **Configuration** for normal use.


## Environment

Use a kernel containing the current MoDaCor checkout with the `server`, `attenuation`, and `plotting` extras plus `hdf5plugin` and `matplotlib`.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/I22")
sys.path.insert(0, str(PROJECT_DIR))

import atexit
import os
import sys

import h5py
import numpy as np
from IPython.display import JSON, Markdown, display

import hdf5plugin
import modacor
from modacor.client import LocalRuntimeServer
from modacor.runner.pipeline import Pipeline

from i22_helpers import (
    DETECTOR_DATASETS,
    build_complete_plan,
    chunk_spec,
    chunk_work_items,
    compare_run_groups,
    direct_sample_registration,
    prepare_inputs,
    sample_aligned_paths,
    source_registrations,
    trace_source_slices,
    upload_sample_chunk,
    validate_chunk_sources,
    validation_pipeline_yaml,
)


## Configuration


In [ ]:
DETECTORS = ("SAXS", "WAXS")
MAX_MEASUREMENTS = None  # use 1 for a quick smoke run
BSDIODES_CHANNEL = 1
# Provisional scalar retained from the DAWN processing record for this example.
ABSOLUTE_INTENSITY_FACTOR = 3.8e-15
OVERWRITE_PREPROCESSED = False

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE = {"enabled": True, "watch": {"sample": ["signal"], "background": ["signal"]}}
OUTPUT_DATA_PATHS = ["/sample/signal", "/sample/Q"]
OUTPUT_DIR = PROJECT_DIR / "work" / "output"


## Prepare inputs

The helper discovers complete sample masters and writes compact files below `work/preprocessed/`. Detector images remain external links; only diode statistics, count times, transmission, and the provisional DAWN scalar are reshaped or summarized.


In [ ]:
inputs = prepare_inputs(
    PROJECT_DIR,
    diode_channel=BSDIODES_CHANNEL,
    absolute_intensity_factor=ABSOLUTE_INTENSITY_FACTOR,
    overwrite=OVERWRITE_PREPROCESSED,
)

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Measurements: {len(inputs.measurements)}")
print(f"Preprocessed data: {inputs.work_dir / 'preprocessed'}")


## Preview the correction graphs


In [ ]:
for detector in DETECTORS:
    pipeline = Pipeline.from_yaml_file(yaml_file=inputs.pipeline_paths[detector])
    pipeline.prepare()
    display(Markdown(f"### {detector}\n\n```mermaid\n{pipeline.to_mermaid(direction='TD')}\n```"))
    print(f"{detector}: {len(pipeline.graph)} steps")


## Start or reuse a local runtime


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=OUTPUT_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Process the batch

`mode="auto"` lets the server choose a full first run when no reusable state exists. Later sample changes reuse the unchanged background and calibration branches, so the notebook does not track server state itself.


In [ ]:
measurements = inputs.measurements[:MAX_MEASUREMENTS]
results = []
sessions = {}

for detector in DETECTORS:
    session = client.replace_session(
        f"i22-{detector.lower()}-server-batch",
        name=f"I22 {detector} server batch",
        pipeline_yaml_path=str(inputs.pipeline_paths[detector]),
        trace=TRACE,
    )
    session.register_sources(*source_registrations(inputs, sample=measurements[0][1]))
    session.register_sink({"ref": "plots", "type": "plotly_json", "location": "buffer://session"})
    sessions[detector] = session

    for master, prepared in measurements:
        output = OUTPUT_DIR / f"{master.stem}_{detector.lower()}_server_result.h5"
        session.register_source({"ref": "sample", "type": "hdf", "location": str(prepared)})
        session.register_sink({"ref": "result_hdf", "type": "hdf", "location": str(output)})
        result = session.process(
            mode="auto",
            changed_sources=["sample"],
            run_name=f"{master.stem}_{detector.lower()}",
            rollback_snapshot=False,
            write_hdf={"path": str(output), "data_paths": OUTPUT_DATA_PATHS},
        )
        results.append({"detector": detector, "measurement": master.name, "output": output, "result": result})
        print(f"{detector} {master.name}: {result['status']} ({result['effective_mode']})")

print(f"Completed {len(results)} detector/measurement runs.")


## Live plots and diagnostics


In [ ]:
links = ["## Latest runtime plots"]
for detector, session in sessions.items():
    links.extend([
        f"### {detector}",
        f"- [Corrected I(Q)]({session.plot_url('plots', detector.lower() + '-1d')})",
        f"- [Corrected detector image]({session.plot_url('plots', detector.lower() + '-2d')})",
    ])
display(Markdown("\n".join(links)))

display(JSON({detector: session.latest_error() for detector, session in sessions.items()}))


## Cleanup

This stops only a process launched by this notebook. A runtime that was already listening on the configured port is left alone.


In [ ]:
server.stop()
print("Stopped the notebook-owned runtime." if not client.is_ready() else "Left the external runtime running.")
